# Pruebas de justificación de coautoría con Gemma 4 12B

Se repiten las cuatro pruebas de Gemma 4 12B con el modo de razonamiento desactivado. Se mantienen los casos, las instrucciones y los parámetros de generación de la primera prueba.

Abre Ollama y ejecuta todas las celdas en orden desde la carpeta donde se encuentran `casos.json`, `inventario_22_llm.csv` y `resultados.csv`. Cada respuesta se guarda al terminar su prueba. Si vuelves a ejecutar toda la libreta, se agregan cuatro intentos nuevos.

## 1. Cargar los casos y seleccionar el modelo

Se probarán los casos C001 y C002, cada uno en español e inglés.

In [ ]:
import os
import csv
import json
import time
from copy import deepcopy
from datetime import datetime, timezone

import requests

ID_MODELO = "M05"
CASOS = ["C001", "C002"]
IDIOMAS = ["es", "en"]

# Cargar los casos
with open("casos.json", "r", encoding="utf-8-sig") as f:
    datos = json.load(f)

casos = {caso["id_caso"]: caso for caso in datos["casos"]}

# Cargar el inventario de modelos
with open("inventario_22_llm.csv", "r", encoding="utf-8-sig", newline="") as f:
    modelos = {fila["id_modelo"]: fila for fila in csv.DictReader(f)}

modelo = modelos[ID_MODELO]

# Revisar que estén disponibles los casos e idiomas
for id_caso in CASOS:
    casos[id_caso]
for idioma in IDIOMAS:
    datos["idiomas"]["instruccion_por_idioma"][idioma]

print("Modelo:", modelo["nombre_modelo"])
print("Etiqueta:", modelo["etiqueta_ollama"])
print("Pruebas:", len(CASOS) * len(IDIOMAS))

## 2. Preparar el mensaje para el LLM

Se conserva la clasificación recibida. Solo se aplica un umbral cuando se dispone de la probabilidad de coautoría y del umbral correspondiente. La referencia de evaluación no se incluye en la solicitud.

In [ ]:
instrucciones = """Genera una justificación semántica de la predicción de coautoría recibida.
Conserva el resultado de la predicción. Si faltan probabilidades por método, indícalo.
Revisa la similitud entre los artículos y el contenido de sus títulos, resúmenes y palabras clave.
Identifica coincidencias y diferencias entre las publicaciones de ambos autores.
Cita el identificador de cada artículo que respalde tus afirmaciones.
Si la evidencia es insuficiente, indica por qué.
Distingue la similitud entre artículos de la probabilidad de coautoría.
No inventes información ni atribuyas causas al método de predicción.
Utiliza únicamente la información proporcionada."""

def preparar_solicitud(caso, idioma):
    # Copiar la entrada para no modificar el caso original
    entrada = deepcopy(caso["entrada_llm"])
    prediccion = entrada["prediccion_coautoria"]
    umbral_general = datos["regla_coautoria"]["umbral_definitivo"]

    for metodo in prediccion["resultados_por_metodo"]:
        p = metodo.get("probabilidad_coautoria")
        umbral = metodo.get("umbral_aplicado")
        if umbral is None:
            umbral = umbral_general

        metodo["umbral_aplicado"] = umbral
        if p is not None and umbral is not None:
            metodo["clasificacion_por_umbral"] = "favorable" if p >= umbral else "desfavorable"
        else:
            metodo["clasificacion_por_umbral"] = None

    instruccion_idioma = datos["idiomas"]["instruccion_por_idioma"][idioma]

    solicitud = {
        "model": modelo["etiqueta_ollama"],
        "messages": [
            {"role": "system", "content": instrucciones + "\n" + instruccion_idioma},
            {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, indent=2)}
        ],
        "stream": False,
        "think": False,
        # Se mantienen los parámetros de la prueba anterior
        "options": {
            "temperature": 0,
            "num_ctx": 4096,
            "num_predict": 1024
        }
    }
    return solicitud

## 3. Consultar Ollama

Se conserva la función de la primera libreta. También se muestra el motivo por el que terminó la generación, la cantidad de tokens y si Ollama devolvió texto en el campo de razonamiento.

In [ ]:
URL_OLLAMA = "http://localhost:11434/api"

def hacer_prueba(solicitud):
    registro = {
        "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "version_ollama": "",
        "digest_modelo": "",
        "solicitud_json": json.dumps(solicitud, ensure_ascii=False),
        "estado": "error",
        "tiempo_s": "",
        "respuesta": "",
        "error": "",
        "evaluacion": "",
        "motivo_evaluacion": ""
    }

    t_inicio = None
    try:
        # Consultar versión de Ollama y digest del modelo
        r_ver = requests.get(URL_OLLAMA + "/version", timeout=10)
        r_ver.raise_for_status()
        registro["version_ollama"] = r_ver.json()["version"]

        r_tags = requests.get(URL_OLLAMA + "/tags", timeout=10)
        r_tags.raise_for_status()
        for local in r_tags.json()["models"]:
            if local["name"].lower() == solicitud["model"].lower():
                registro["digest_modelo"] = local.get("digest", "")
                break

        # Enviar solicitud al modelo
        t_inicio = time.time()
        r_chat = requests.post(URL_OLLAMA + "/chat", json=solicitud, timeout=900)
        if not r_chat.ok:
            registro["error"] = f"Ollama devolvió HTTP {r_chat.status_code}: {r_chat.text}"
        else:
            salida = r_chat.json()
            print("Motivo de término:", salida.get("done_reason"))
            print("Tokens generados:", salida.get("eval_count"))
            print("Tokens de entrada:", salida.get("prompt_eval_count"))
            print("Razonamiento recibido:", "Sí" if salida["message"].get("thinking") else "No")
            texto = salida["message"]["content"]
            registro["respuesta"] = texto

            if salida.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif salida.get("done") and texto.strip() != "":
                registro["estado"] = "completa"
            else:
                registro["estado"] = "incompleta"

    except Exception as e:
        registro["error"] = str(e)

    if t_inicio is not None:
        registro["tiempo_s"] = round(time.time() - t_inicio, 2)

    return registro

## 4. Preparar el archivo de resultados

Se utilizan las mismas columnas y se conserva la numeración de intentos por caso y modelo. No se eliminan los resultados de las pruebas anteriores.

In [ ]:
archivo_resultados = "resultados.csv"
columnas = [
    "id_caso", "id_modelo", "idioma", "intento", "fecha_utc", "version_ollama", "digest_modelo",
    "solicitud_json", "estado", "tiempo_s", "respuesta", "error", "evaluacion", "motivo_evaluacion"
]

# Leer los intentos anteriores
filas = []
if os.path.exists(archivo_resultados) and os.path.getsize(archivo_resultados) > 0:
    with open(archivo_resultados, "r", encoding="utf-8-sig", newline="") as f:
        lector = csv.DictReader(f)
        if lector.fieldnames != columnas:
            raise ValueError("Revisa los encabezados de resultados.csv antes de continuar.")
        filas = list(lector)
else:
    with open(archivo_resultados, "w", encoding="utf-8-sig", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writeheader()

print("Intentos anteriores:", len(filas))

def guardar_resultado(registro, id_caso, idioma):
    # Continuar la numeración de cada caso y modelo
    intento = 1
    for fila in filas:
        if fila["id_caso"] == id_caso and fila["id_modelo"] == ID_MODELO:
            intento = max(intento, int(fila["intento"]) + 1)

    registro["id_caso"] = id_caso
    registro["id_modelo"] = ID_MODELO
    registro["idioma"] = idioma
    registro["intento"] = intento

    with open(archivo_resultados, "a", encoding="utf-8-sig", newline="") as f:
        escritor = csv.DictWriter(f, fieldnames=columnas)
        escritor.writerow(registro)

    filas.append(registro)

## 5. Ejecutar las cuatro pruebas

Se consulta el modelo una vez por cada caso e idioma, sin cambiar las variables manualmente. Antes de empezar se revisa que el modelo esté instalado. Al terminar cada prueba se guarda su resultado, incluso si Ollama devuelve un error.

In [ ]:
# Revisar que Ollama esté abierto y tenga el modelo
r_tags = requests.get(URL_OLLAMA + "/tags", timeout=10)
r_tags.raise_for_status()
etiquetas = [local["name"].lower() for local in r_tags.json()["models"]]
if modelo["etiqueta_ollama"].lower() not in etiquetas:
    raise ValueError("El modelo no está instalado: " + modelo["etiqueta_ollama"])

# Ejecutar cada caso en los dos idiomas
for id_caso in CASOS:
    for idioma in IDIOMAS:
        print("\n" + "-" * 60)
        print(id_caso, "|", modelo["nombre_modelo"], "|", idioma, flush=True)

        solicitud = preparar_solicitud(casos[id_caso], idioma)
        registro = hacer_prueba(solicitud)
        guardar_resultado(registro, id_caso, idioma)

        print("Intento:", registro["intento"], "| Estado:", registro["estado"],
              "| Tiempo (s):", registro["tiempo_s"])
        print(registro["error"] if registro["error"] else registro["respuesta"])

print("\nPruebas terminadas. Resultados guardados en", archivo_resultados)

## 6. Revisar las justificaciones

Se compara cada respuesta con los artículos del caso. La evaluación se anota en las columnas `evaluacion` y `motivo_evaluacion` de `resultados.csv`. Una respuesta completa no necesariamente es una justificación correcta.

In [ ]:
for id_caso in CASOS:
    referencia = casos[id_caso]["referencia_evaluacion"]
    print("\nCaso:", id_caso)
    print("Motivo de selección:", casos[id_caso]["motivo_de_seleccion"])
    print("Artículos de referencia:", ", ".join(referencia["articulos"]))
    print("Afirmación respaldada:", referencia["afirmacion_respaldada"])
    print("Límite de la evidencia:", referencia["limite"])